# Identifying Causal Effects using Z-Identifiability (Surrogate Experiments)

This notebook demonstrates **z-identifiability** in DoWhy — a method for identifying causal effects in the presence of hidden confounding, using surrogate experiments (Bareinboim & Pearl, 2012).

**When to use z-identifiability:**
Standard identification methods (backdoor, frontdoor, IV) fail when there is unblockable hidden confounding between treatment and outcome. Z-identifiability rescues identification when a *surrogate variable* Z exists — one that can be intervened on in an auxiliary experiment — allowing estimation of P(Y | do(X)) via:

> P(Y | do(X)) = Σ_z P(Y | X, Z) P(Z)

**Installation:**
```bash
pip install dowhy[zid]
```

**Reference:** Bareinboim, E. & Pearl, J. (2012). *Causal Inference by Surrogate Experiments: z-Identifiability.* UAI.

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd

from dowhy.causal_identifier.auto_identifier import EstimandType, identify_effect_auto
from dowhy.causal_identifier.zid_identifier import ZIDIdentifier

## Helper: Building DoWhy-style graphs

DoWhy represents latent (hidden) confounders as unobserved nodes with `observed="no"`.
Each such node with two children encodes a bidirected arc (hidden common cause) between them.

In [ ]:
def build_graph(di_edges, hidden_confounders=None):
    """
    Build a DoWhy-style nx.DiGraph.
    
    Parameters
    ----------
    di_edges : list of (str, str)
        Directed edges among observed variables.
    hidden_confounders : list of (str, str), optional
        Each tuple (A, B) adds a hidden common cause between A and B,
        encoded as an unobserved node U_i -> A, U_i -> B.
    """
    G = nx.DiGraph()
    observed = set()
    for u, v in di_edges:
        G.add_edge(u, v)
        observed.update([u, v])
    for n in observed:
        G.nodes[n]["observed"] = "yes"
    for i, (a, b) in enumerate(hidden_confounders or []):
        uid = f"U_{i}"
        G.add_node(uid, observed="no")
        G.add_edge(uid, a)
        G.add_edge(uid, b)
    return G

## Example 1: Standard identification fails, z-ID rescues

Consider a 5-variable graph where:
- Z → X → Y (Z is a surrogate that causes treatment X)
- Hidden confounders create bidirected arcs: W₁↔X, W₁↔Y, W₂↔Z, X↔Z, Y↔Z

Standard identification methods (backdoor, IV, frontdoor) all fail here due to the unblockable hidden confounding. However, z-identifiability succeeds using Z as a surrogate.

**Identifying functional:** P(Y | do(X)) = Σ_z P(Y | X, Z) P(Z)

In [ ]:
# Build the graph
G_rescue = build_graph(
    di_edges=[("W_1", "Z"), ("Z", "X"), ("X", "Y")],
    hidden_confounders=[
        ("W_1", "X"),   # W_1 <-> X
        ("W_1", "Y"),   # W_1 <-> Y
        ("W_2", "Z"),   # W_2 <-> Z
        ("X", "Z"),     # X <-> Z
        ("Y", "Z"),     # Y <-> Z
    ]
)

observed_nodes = [n for n in G_rescue.nodes if G_rescue.nodes[n].get("observed", "yes") == "yes"]
print("Observed variables:", observed_nodes)
print("Hidden confounders: W_1<->X, W_1<->Y, W_2<->Z, X<->Z, Y<->Z")

In [ ]:
# Run identification via identify_effect_auto with surrogate_nodes
estimand = identify_effect_auto(
    G_rescue,
    action_nodes=["X"],
    outcome_nodes=["Y"],
    observed_nodes=observed_nodes,
    estimand_type=EstimandType.NONPARAMETRIC_ATE,
    surrogate_nodes=["Z"],   # Z is our surrogate variable
)

print("Backdoor estimand:   ", estimand.estimands.get("backdoor"))
print("IV estimand:         ", estimand.estimands.get("iv"))
print("Frontdoor estimand:  ", estimand.estimands.get("frontdoor"))
print()
print("Z-ID estimand:       ", estimand.estimands.get("zid"))
print("Surrogate variables: ", estimand.estimands["zid"].backdoor_variables if estimand.estimands.get("zid") else None)

The output shows that **backdoor, IV, and frontdoor all return None** — standard identification fails. But `zid` succeeds, with `backdoor_variables=["Z"]` encoding the surrogate adjustment.

## Example 2: Using ZIDIdentifier directly

For more control, you can use `ZIDIdentifier` directly without going through `identify_effect_auto`.

In [ ]:
# Direct ZIDIdentifier usage
zid = ZIDIdentifier(
    graph=G_rescue,
    action_nodes=["X"],
    outcome_nodes=["Y"],
    surrogate_nodes=["Z"],
)

estimand_direct = zid.identify_effect()
print("Z-ID succeeded:", estimand_direct is not None)
print("Surrogate adjustment set:", estimand_direct.backdoor_variables)
print()
print("Identifying functional: P(Y | do(X)) = Σ_z P(Y | X, Z) P(Z)")

## Example 3: Non-identifiable case

Z-identifiability has limits. If the surrogate Z is itself confounded with the outcome Y (Z↔Y bidirected arc), it can no longer rescue identification.

In [ ]:
# Graph where z-ID also fails: X->Y with X<->Y and Z<->Y
G_fail = build_graph(
    di_edges=[("Z", "X"), ("X", "Y")],
    hidden_confounders=[
        ("X", "Y"),   # X <-> Y  (unblockable)
        ("Z", "Y"),   # Z <-> Y  (surrogate confounded with outcome)
    ]
)

observed_fail = [n for n in G_fail.nodes if G_fail.nodes[n].get("observed", "yes") == "yes"]

try:
    zid_fail = ZIDIdentifier(G_fail, ["X"], ["Y"], ["Z"])
    zid_fail.identify_effect()
    print("Identified (unexpected)")
except Exception as e:
    print(f"Not z-identifiable: {e}")

## Example 4: Standard identification — no surrogates needed

Z-identifiability is a superset of standard identification. When no hidden confounding exists, it correctly identifies via standard methods and `zid` is None (surrogates not needed).

In [ ]:
# Clean graph: Z -> X -> Y, no hidden confounders
G_clean = build_graph(
    di_edges=[("Z", "X"), ("X", "Y")],
    hidden_confounders=[]
)

estimand_clean = identify_effect_auto(
    G_clean,
    action_nodes=["X"],
    outcome_nodes=["Y"],
    observed_nodes=["Z", "X", "Y"],
    estimand_type=EstimandType.NONPARAMETRIC_ATE,
    # No surrogate_nodes — standard methods apply
)

print("Backdoor estimand:  ", estimand_clean.estimands.get("backdoor"))
print("Z-ID estimand:      ", estimand_clean.estimands.get("zid"))  # None — not needed

## Summary

| Scenario | Backdoor | IV | Frontdoor | Z-ID |
|---|---|---|---|---|
| Clean graph (no confounding) | ✓ | — | — | N/A |
| Hidden confounding, valid surrogate | ✗ | ✗ | ✗ | ✓ |
| Hidden confounding, surrogate also confounded | ✗ | ✗ | ✗ | ✗ |

**Key API:**
```python
# Via identify_effect_auto (recommended)
estimand = identify_effect_auto(graph, ["X"], ["Y"], observed, 
                                 EstimandType.NONPARAMETRIC_ATE,
                                 surrogate_nodes=["Z"])
estimand.estimands["zid"]  # IdentifiedEstimand or None

# Via ZIDIdentifier directly
zid = ZIDIdentifier(graph, ["X"], ["Y"], surrogate_nodes=["Z"])
estimand = zid.identify_effect()  # raises if not z-identifiable
```

**Reference:** Bareinboim, E. & Pearl, J. (2012). *Causal Inference by Surrogate Experiments: z-Identifiability.* UAI.  
**Package:** `pip install dowhy[zid]` (requires `pyananke >= 0.6.1`)